In [ ]:
import os
import re
import csv
import numpy as np
import pandas as pd
import mendeleev as md
from math import gcd
from ase.io import read
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 20000)
pd.set_option('display.expand_frame_repr', True)

root = '/Users/jiuy97/Desktop/3_RuO2/7_prediction'
if not os.path.exists(root):
    root = '/Users/hailey/Desktop/3_RuO2/7_prediction'
oxygen_potential = -4.658724749999999+0.27-0.73  # 700K
figsize = (8, 6)
dpi = 300

In [ ]:
element_indices = list(range(21, 31)) + list(range(39, 49)) + [57] + list(range(72, 81))
elements = [md.element(i).symbol for i in element_indices]
data = pd.DataFrame(index=elements)
data['a'] = element_indices

In [ ]:
for i in element_indices:
    element = md.element(i).symbol
    for dir in ['0_M', '1_MxOy', '2_MO2', '3_M-RuO2', '4_M-IrO2']:
        oxide_type = dir.split('_')[1]
        path = os.path.join(root, '1_bulk_opt', dir, f'{i}_{element}')
        energy_file = os.path.join(path, 'final_with_calculator.json')
        moment_file = os.path.join(path, 'moments.json')
        charge_file = os.path.join(path, 'atoms_bader_charge.json')
        row_updates = {}
        if os.path.exists(energy_file):
            atoms = read(energy_file)
            metal_indices = [i for i, atom in enumerate(atoms) if atom.symbol in elements]
            n_atoms = len(atoms)
            n_oxygens = sum(atom.symbol == 'O' for atom in atoms)
            n_metals = sum(atom.symbol != 'O' for atom in atoms)
            oxidation_state = 2 * n_oxygens / n_metals
            energy = atoms.get_potential_energy() / n_metals
            row_updates[f'{oxide_type}_e'] = energy
            if oxide_type == 'MxOy':
                common_divisor = gcd(n_metals, n_oxygens)
                m_ratio = n_metals // common_divisor
                o_ratio = n_oxygens // common_divisor
                row_updates['MxOy_x'] = m_ratio
                row_updates['MxOy_y'] = o_ratio
                row_updates['MxOy_os'] = oxidation_state
        if os.path.exists(moment_file):
            atoms = read(moment_file)
            moments = atoms.get_magnetic_moments()
            if oxide_type == 'M-RuO2':
                row_updates['M-RuO2_m'] = np.mean(np.abs(moments[0:7]))
                for j, moment in enumerate(moments[metal_indices]):
                    row_updates[f'M-RuO2_m{j}'] = moment
            elif oxide_type == 'M-IrO2':
                row_updates['M-IrO2_m'] = np.mean(np.abs(moments[0:7]))
                for j, moment in enumerate(moments[metal_indices]):
                    row_updates[f'M-IrO2_m{j}'] = moment
            else:
                row_updates[f'{oxide_type}_m'] = np.mean(np.abs(moments[metal_indices]))
        if os.path.exists(charge_file):
            atoms = read(charge_file)
            charges = atoms.get_initial_charges()
            if oxide_type == 'M-RuO2':
                row_updates['M-RuO2_c'] = np.mean(np.abs(charges[0:7]))
                for j, charge in enumerate(charges):
                    row_updates[f'M-RuO2_c{j}'] = charge
            elif oxide_type == 'M-IrO2':
                row_updates['M-IrO2_c'] = np.mean(np.abs(charges[0:7]))
                for j, charge in enumerate(charges):
                    row_updates[f'M-IrO2_c{j}'] = charge
            else:
                row_updates[f'{oxide_type}_c'] = np.mean(np.abs(charges[metal_indices]))
        if row_updates:
            data.loc[element, list(row_updates)] = list(row_updates.values())

data['M-RuO2_fe1'] = data['M-RuO2_e'] - 7/8*data['M_e']['Ru'] - 1/8*data['M_e'] - 2*oxygen_potential
data['M-RuO2_fe2'] = data['M-RuO2_e'] - 7/8*data['MO2_e']['Ru'] - 1/8*data['MO2_e']
data['M-RuO2_fe3'] = data['M-RuO2_e'] - 7/8*data['MxOy_e']['Ru'] - 1/8*data['MxOy_e'] - (2-data['MxOy_y']/data['MxOy_x'])/8*oxygen_potential
data['M-IrO2_fe1'] = data['M-IrO2_e'] - 7/8*data['M_e']['Ir'] - 1/8*data['M_e'] - 2*oxygen_potential
data['M-IrO2_fe2'] = data['M-IrO2_e'] - 7/8*data['MO2_e']['Ir'] - 1/8*data['MO2_e']
data['M-IrO2_fe3'] = data['M-IrO2_e'] - 7/8*data['MxOy_e']['Ir'] - 1/8*data['MxOy_e'] - (2-data['MxOy_y']/data['MxOy_x'])/8*oxygen_potential
data.drop(columns=['MO2_e', 'MxOy_e', 'M-RuO2_e', 'M-IrO2_e'], inplace=True)
data.loc['Nb', 'MxOy_x'] = 2
data.loc['Nb', 'MxOy_y'] = 5
data

In [ ]:
# ── Fancy style settings ───────────────────────────────────────────────────
import matplotlib as mpl

mpl.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.labelsize':    12,
    'axes.titlesize':    12,
    'xtick.labelsize':   9.5,
    'ytick.labelsize':   10,
    'legend.fontsize':   9.5,
    'axes.linewidth':    0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.major.size':  4,
    'ytick.major.size':  4,
})

# Paul Tol's colorblind-friendly bright palette
palette = ['#0077BB', '#EE7733', '#009988']   # 3d, 4d, 5d
MARKERS = ['o', 's', '^']
MSIZE   = 7
LW      = 1.8

def style_ax(ax):
    """Remove top/right spines and add a subtle y-grid."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle=':', linewidth=0.6, alpha=0.45, zorder=0)
    ax.set_axisbelow(True)

def add_period_bands(ax, slices, n, colors, alpha=0.08):
    """Shade x-axis regions for 3d/4d/5d transition metals."""
    offset = slices[0][0]
    for si, (lo, hi) in enumerate(slices):
        if lo >= n:
            break
        ax.axvspan(lo - offset + 0.5, min(hi, n) - offset + 0.5,
                   color=colors[si], alpha=alpha, zorder=0, linewidth=0)

# ── Shared axis setup ─────────────────────────────────────────────────────
n = len(data)
chunk = 10
slices = [(0, chunk), (chunk, 2 * chunk), (2 * chunk, n)]
row_suffix = ['3d', '4d', '5d']
x = np.arange(1, chunk + 1)
xtick_labels = []
for i in range(chunk):
    xtick_labels.append(
        '\n'.join(
            str(data.index[lo + i])
            for lo, hi in slices
            if lo + i < min(hi, n)
        )
    )

In [ ]:
# ── Formation energy (fancy) ───────────────────────────────────────────────
ref_title = {
    '1': 'ref: M (metal) + O$_2$',
    '2': 'ref: MO$_2$ (rutile)',
    '3': 'ref: M$_x$O$_y$ + O$_2$',
}

for col in ['M-RuO2_fe1', 'M-RuO2_fe2', 'M-RuO2_fe3',
            'M-IrO2_fe1', 'M-IrO2_fe2', 'M-IrO2_fe3']:
    host_oxide = col.split('M-')[1].split('_')[0]
    c = col[-1]

    fig, ax = plt.subplots(figsize=(5, 3.5))
    add_period_bands(ax, slices, n, palette)

    for si, (lo, hi) in enumerate(slices):
        if lo >= n:
            break
        hi = min(hi, n)
        y = (data[col].iloc[lo:hi] * 8).to_numpy()
        ax.plot(
            x[:len(y)], y,
            marker=MARKERS[si], markersize=MSIZE,
            markeredgecolor='white', markeredgewidth=0.7,
            linestyle='-', linewidth=LW,
            label=row_suffix[si], color=palette[si], zorder=2,
        )

    if c != '1':
        ax.axhline(0, color='#555555', linestyle='--',
                   linewidth=0.9, zorder=1, alpha=0.7)

    ax.set_xlim(0.5, chunk + 0.5)
    if c == '2':
        ax.set_ylim(-3.0, 1.5)
    elif c == '3':
        ax.set_ylim(-1.0, 3.0)

    ax.set_xticks(x)
    ax.set_xticklabels(xtick_labels)
    ax.set_xlabel('Dopant (M)', labelpad=6)
    ax.set_ylabel('Formation energy (eV/dopant)', labelpad=6)

    ax.legend(
        title=ref_title[c], title_fontsize=8.5,
        loc='upper center', bbox_to_anchor=(0.5, 1.18),
        ncol=3, frameon=True, framealpha=0.92, edgecolor='#cccccc',
    )
    style_ax(ax)
    fig.tight_layout()
    fig.savefig(f'{root}/figures/bulk_{host_oxide}_formation_energy{c}_fancy.png',
                dpi=dpi, transparent=True, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# ── Magnetic moment (fancy) ────────────────────────────────────────────────
mm_titles = {
    'M_m':       'Pure metal',
    'MxOy_m':    'M$_x$O$_y$',
    'MO2_m':     'MO$_2$ (rutile)',
    'M-RuO2_m7': 'M-doped RuO$_2$',
    'M-IrO2_m7': 'M-doped IrO$_2$',
}

for c, col in enumerate(['M_m', 'MxOy_m', 'MO2_m', 'M-RuO2_m7', 'M-IrO2_m7']):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    add_period_bands(ax, slices, n, palette)

    for si, (lo, hi) in enumerate(slices):
        if lo >= n:
            break
        hi = min(hi, n)
        y = np.abs(data[col].iloc[lo:hi]).to_numpy()
        ax.plot(
            x[:len(y)], y,
            marker=MARKERS[si], markersize=MSIZE,
            markeredgecolor='white', markeredgewidth=0.7,
            linestyle='-', linewidth=LW,
            label=row_suffix[si], color=palette[si], zorder=2,
        )

    ax.set_xlim(0.5, chunk + 0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(xtick_labels)
    ax.set_xlabel('Dopant (M)', labelpad=6)
    ax.set_ylabel('Magnetic moment (\u03bc$_B$/dopant)', labelpad=6)
    ax.set_title(mm_titles[col], fontsize=11, pad=6)

    ax.legend(
        loc='upper center', bbox_to_anchor=(0.5, 1.18),
        ncol=3, frameon=True, framealpha=0.92, edgecolor='#cccccc',
    )
    style_ax(ax)
    fig.tight_layout()
    fig.savefig(f'{root}/figures/bulk_magnetic_moment{c}_fancy.png',
                dpi=dpi, transparent=True, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# ── Bader charge (fancy) ───────────────────────────────────────────────────
bc_titles = {
    'M_c':       'Pure metal',
    'MxOy_c':    'M$_x$O$_y$',
    'MO2_c':     'MO$_2$ (rutile)',
    'M-RuO2_c7': 'M-doped RuO$_2$',
    'M-IrO2_c7': 'M-doped IrO$_2$',
}

for c, col in enumerate(['M_c', 'MxOy_c', 'MO2_c', 'M-RuO2_c7', 'M-IrO2_c7']):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    add_period_bands(ax, slices, n, palette)

    for si, (lo, hi) in enumerate(slices):
        if lo >= n:
            break
        hi = min(hi, n)
        y = np.abs(data[col].iloc[lo:hi]).to_numpy()
        ax.plot(
            x[:len(y)], y,
            marker=MARKERS[si], markersize=MSIZE,
            markeredgecolor='white', markeredgewidth=0.7,
            linestyle='-', linewidth=LW,
            label=row_suffix[si], color=palette[si], zorder=2,
        )

    ax.set_xlim(0.5, chunk + 0.5)
    ax.set_ylim(0.0, 3.0)
    ax.set_xticks(x)
    ax.set_xticklabels(xtick_labels)
    ax.set_xlabel('Dopant (M)', labelpad=6)
    ax.set_ylabel('Bader charge (e/dopant)', labelpad=6)
    ax.set_title(bc_titles[col], fontsize=11, pad=6)

    ax.legend(
        loc='upper center', bbox_to_anchor=(0.5, 1.18),
        ncol=3, frameon=True, framealpha=0.92, edgecolor='#cccccc',
    )
    style_ax(ax)
    fig.tight_layout()
    fig.savefig(f'{root}/figures/bulk_bader_charge{c}_fancy.png',
                dpi=dpi, transparent=True, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# ── Convex hull – M-doped RuO2 (fancy) ────────────────────────────────────
HOST_COLOR  = '#2ecc71'   # green  – host MO2 (rutile)
DOPANT_COLOR = '#3498db'  # blue   – dopant reference MxOy
POINT_COLOR  = '#e74c3c'  # red    – doped system (diamond)

for i in element_indices:
    element = md.element(i).symbol
    m_ratio = int(data.loc[element, 'MxOy_x'])
    o_ratio = int(data.loc[element, 'MxOy_y'])
    if m_ratio == 1 and o_ratio == 1:
        formula = f'{element}O'
    elif m_ratio == 1:
        formula = f'{element}O$_{o_ratio}$'
    else:
        formula = f'{element}$_{m_ratio}$O$_{o_ratio}$'

    fe2 = data.loc[element, 'M-RuO2_fe2']
    fe3 = data.loc[element, 'M-RuO2_fe3']
    delta = 8 * (fe2 - fe3)   # energy difference at x=1

    fig, ax = plt.subplots(figsize=(4.5, 3.5))

    # Hull: MO2 reference (flat)
    ax.plot([0.0, 1.0], [0.0, 0.0],
            color='#333333', linewidth=1.6,
            marker='s', markersize=8,
            markeredgecolor='white', markeredgewidth=0.9,
            markerfacecolor=HOST_COLOR, zorder=3)

    # Hull: MxOy reference (sloped)
    ax.plot([0.0, 1.0], [0.0, delta],
            color='#333333', linewidth=1.6,
            marker='s', markersize=8,
            markeredgecolor='white', markeredgewidth=0.9,
            markerfacecolor=DOPANT_COLOR, zorder=3)

    # Fill hull area
    ax.fill_between([0.0, 1.0], [0.0, 0.0], [0.0, delta],
                    alpha=0.12, color=DOPANT_COLOR, zorder=0)

    # Doped system point
    ax.scatter(1/8, fe2, marker='D', s=75,
               edgecolors='#333333', facecolors=POINT_COLOR,
               linewidth=1.0, zorder=4)

    # Host label
    ax.text(0.0, 0.08, 'RuO$_2$\n(rutile)',
            ha='center', va='bottom', fontsize=8.5,
            linespacing=1.2, color=HOST_COLOR, fontweight='semibold')

    # Dopant labels
    if delta > 0:
        ax.text(1.0, delta + 0.10, formula,
                ha='center', va='bottom', fontsize=8.5,
                color=DOPANT_COLOR, fontweight='semibold')
        ax.text(1.0, -0.10, f'{element}O$_2$\n(rutile)',
                ha='center', va='top', fontsize=8.5,
                linespacing=1.2, color=HOST_COLOR)
    else:
        ax.text(1.0, delta - 0.10, formula,
                ha='center', va='top', fontsize=8.5,
                color=DOPANT_COLOR, fontweight='semibold')
        ax.text(1.0, 0.10, f'{element}O$_2$\n(rutile)',
                ha='center', va='bottom', fontsize=8.5,
                linespacing=1.2, color=HOST_COLOR)

    # Info annotation box (bottom-left corner)
    info = (
        f'$E_f$({element}O$_2$) = {fe2:.2f} eV\n'
        f'$E_f$({formula}) = {fe3:.2f} eV\n'
        f'\u03bc$_O$ = \u22120.73 eV (700 K)'
    )
    ax.text(0.03, 0.03, info, transform=ax.transAxes,
            ha='left', va='bottom', fontsize=7.5, linespacing=1.45,
            bbox=dict(boxstyle='round,pad=0.45', fc='white',
                      ec='#cccccc', alpha=0.88, linewidth=0.7))

    ax.set_xlim(-0.12, 1.12)
    ax.set_ylim(-3.0, 2.0)
    ax.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_xlabel('Dopant fraction (M)', labelpad=6)
    ax.set_ylabel('Formation energy (eV/metal)', labelpad=6)
    ax.set_title(f'{element} in RuO$_2$', fontsize=12, pad=6, fontweight='semibold')
    style_ax(ax)
    fig.tight_layout()
    fig.savefig(f'{root}/figures/bulk_convex_hull_RuO2_{i}{element}_fancy.png',
                dpi=dpi, transparent=True, bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
# ── Convex hull – M-doped IrO2 (fancy) ────────────────────────────────────
for i in element_indices:
    element = md.element(i).symbol
    m_ratio = int(data.loc[element, 'MxOy_x'])
    o_ratio = int(data.loc[element, 'MxOy_y'])
    if m_ratio == 1 and o_ratio == 1:
        formula = f'{element}O'
    elif m_ratio == 1:
        formula = f'{element}O$_{o_ratio}$'
    else:
        formula = f'{element}$_{m_ratio}$O$_{o_ratio}$'

    fe2 = data.loc[element, 'M-IrO2_fe2']
    fe3 = data.loc[element, 'M-IrO2_fe3']
    delta = 8 * (fe2 - fe3)

    fig, ax = plt.subplots(figsize=(4.5, 3.5))

    ax.plot([0.0, 1.0], [0.0, 0.0],
            color='#333333', linewidth=1.6,
            marker='s', markersize=8,
            markeredgecolor='white', markeredgewidth=0.9,
            markerfacecolor=HOST_COLOR, zorder=3)

    ax.plot([0.0, 1.0], [0.0, delta],
            color='#333333', linewidth=1.6,
            marker='s', markersize=8,
            markeredgecolor='white', markeredgewidth=0.9,
            markerfacecolor=DOPANT_COLOR, zorder=3)

    ax.fill_between([0.0, 1.0], [0.0, 0.0], [0.0, delta],
                    alpha=0.12, color=DOPANT_COLOR, zorder=0)

    ax.scatter(1/8, fe2, marker='D', s=75,
               edgecolors='#333333', facecolors=POINT_COLOR,
               linewidth=1.0, zorder=4)

    ax.text(0.0, 0.08, 'IrO$_2$\n(rutile)',
            ha='center', va='bottom', fontsize=8.5,
            linespacing=1.2, color=HOST_COLOR, fontweight='semibold')

    if delta > 0:
        ax.text(1.0, delta + 0.10, formula,
                ha='center', va='bottom', fontsize=8.5,
                color=DOPANT_COLOR, fontweight='semibold')
        ax.text(1.0, -0.10, f'{element}O$_2$\n(rutile)',
                ha='center', va='top', fontsize=8.5,
                linespacing=1.2, color=HOST_COLOR)
    else:
        ax.text(1.0, delta - 0.10, formula,
                ha='center', va='top', fontsize=8.5,
                color=DOPANT_COLOR, fontweight='semibold')
        ax.text(1.0, 0.10, f'{element}O$_2$\n(rutile)',
                ha='center', va='bottom', fontsize=8.5,
                linespacing=1.2, color=HOST_COLOR)

    info = (
        f'$E_f$({element}O$_2$) = {fe2:.2f} eV\n'
        f'$E_f$({formula}) = {fe3:.2f} eV\n'
        f'\u03bc$_O$ = \u22120.73 eV (700 K)'
    )
    ax.text(0.03, 0.03, info, transform=ax.transAxes,
            ha='left', va='bottom', fontsize=7.5, linespacing=1.45,
            bbox=dict(boxstyle='round,pad=0.45', fc='white',
                      ec='#cccccc', alpha=0.88, linewidth=0.7))

    ax.set_xlim(-0.12, 1.12)
    ax.set_ylim(-3.0, 2.0)
    ax.set_xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_xlabel('Dopant fraction (M)', labelpad=6)
    ax.set_ylabel('Formation energy (eV/metal)', labelpad=6)
    ax.set_title(f'{element} in IrO$_2$', fontsize=12, pad=6, fontweight='semibold')
    style_ax(ax)
    fig.tight_layout()
    fig.savefig(f'{root}/figures/bulk_convex_hull_IrO2_{i}{element}_fancy.png',
                dpi=dpi, transparent=True, bbox_inches='tight')
    plt.show()
    plt.close()